# Notebook 02 — Preprocessing & Feature Engineering

**Objective:** Transform raw `profiles.csv` into a numeric feature matrix
`X_processed.npy` that K-Means and PCA can consume.

**Output files produced by this notebook:**
- `data/processed/X_processed.npy` — the feature matrix, shape (50000, N)
- `data/mappings/anon_id_map.csv` — row index to profile_id mapping
- `data/processed/df_model.csv` — intermediate cleaned dataframe (for inspection)

**What is deliberately excluded from the feature matrix:**
- `name`, `email`, `profile_id` — PII identifiers
- `source` — constant column, zero variance
- `headline`, `about` — generic boilerplate, low signal
- Anything from `compatibility_pairs.csv` — never loaded

In [1]:
import os
import sys

NOTEBOOK_DIR = os.path.abspath("")
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root : {PROJECT_ROOT}")
print(f"src exists   : {os.path.isdir(os.path.join(PROJECT_ROOT, 'src'))}")
print(f"data exists  : {os.path.isdir(os.path.join(PROJECT_ROOT, 'data'))}")

Project root : /Users/yugjain/Documents/ML_ASS/unsupervised-professional-matching
src exists   : True
data exists  : True


In [2]:
import ast
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

from src.config import (
    RAW_DATA_PATH, PROCESSED_PATH, MAPPINGS_PATH,
    AST_COLS, SENIORITY_MAP, REMOTE_MAP,
    N_SKILLS, N_TFIDF_TERMS
)
from src.preprocessing.drop_columns import drop_pii_and_leakage
from src.preprocessing.id_mapper import create_id_map

warnings.filterwarnings("ignore")
print("✓ Imports complete")

✓ Imports complete


## 1. Load & Preserve Original

In [3]:
# df_original is the sacred copy — full details, all columns intact
# It is ONLY used by the display layer in notebook 03
# It is NEVER passed into any encoding or scaling function
df_original = pd.read_csv(RAW_DATA_PATH)

print(f"Loaded : {df_original.shape[0]:,} rows × {df_original.shape[1]} columns")
print(f"Columns: {df_original.columns.tolist()}")

Loaded : 50,000 rows × 20 columns
Columns: ['profile_id', 'name', 'email', 'current_role', 'current_company', 'industry', 'years_experience', 'seniority_level', 'skills', 'experience', 'education', 'connections', 'goals', 'needs', 'can_offer', 'location', 'remote_preference', 'headline', 'about', 'source']


In [4]:
# Must happen BEFORE dropping profile_id
# Once profile_id is dropped from df_model, this mapping
# is the only way to recover full profile details from a row index
id_map = create_id_map(df_original)
id_map.head()

✓ ID map saved → /Users/yugjain/Documents/ML_ASS/unsupervised-professional-matching/data/mappings/anon_id_map.csv
  50,000 profiles mapped


,anon_index,profile_id
0,0,PROF-00001
1,1,PROF-00002
2,2,PROF-00003
3,3,PROF-00004
4,4,PROF-00005


## 2. Drop PII & Low-Signal Columns

In [5]:
df_model = drop_pii_and_leakage(df_original)

print(f"\nShape after dropping : {df_model.shape}")
print(f"Remaining columns    : {df_model.columns.tolist()}")

Dropping 6 columns: ['name', 'email', 'profile_id', 'source', 'headline', 'about']

Shape after dropping : (50000, 14)
Remaining columns    : ['current_role', 'current_company', 'industry', 'years_experience', 'seniority_level', 'skills', 'experience', 'education', 'connections', 'goals', 'needs', 'can_offer', 'location', 'remote_preference']


## 3. Null Handling

Nulls must be filled before encoding.
Encoders passed a NaN will either crash or silently produce
incorrect output — filling first prevents both failure modes.

Strategy:
- Numeric columns → fill with column median
- Categorical/text columns → fill with the string "unknown"
- AST columns (JSON fields) → fill with an empty list string "[]"

In [6]:
# Numeric columns — fill with median
numeric_cols = ["years_experience", "connections"]
for col in numeric_cols:
    median_val = df_model[col].median()
    null_count = df_model[col].isna().sum()
    df_model[col] = df_model[col].fillna(median_val)
    if null_count > 0:
        print(f"  {col}: filled {null_count} nulls with median={median_val:.1f}")

# Categorical columns — fill with "unknown"
cat_cols = ["current_role", "industry", "seniority_level",
            "remote_preference", "location"]
for col in cat_cols:
    null_count = df_model[col].isna().sum()
    df_model[col] = df_model[col].fillna("unknown")
    if null_count > 0:
        print(f"  {col}: filled {null_count} nulls with 'unknown'")

# AST/JSON columns — fill with empty list string
for col in AST_COLS:
    null_count = df_model[col].isna().sum()
    df_model[col] = df_model[col].fillna("[]")
    if null_count > 0:
        print(f"  {col}: filled {null_count} nulls with '[]'")

print("\n✓ Null handling complete")
print(f"Remaining nulls: {df_model.isnull().sum().sum()}")


✓ Null handling complete
Remaining nulls: 0


## 4. Parse AST Fields

All JSON-like columns are stored as Python literal strings using single quotes.
`ast.literal_eval()` is required — `json.loads()` will fail on single-quoted strings.
This was confirmed during EDA in notebook 01.

In [7]:
def safe_parse(val):
    """
    Parse a Python literal string into a Python object.
    Uses ast.literal_eval — NOT json.loads.
    Returns an empty list on failure or null input.
    """
    if pd.isna(val) or val == "[]":
        return []
    if isinstance(val, (list, dict)):
        return val
    try:
        result = ast.literal_eval(val)
        return result if result is not None else []
    except Exception:
        return []

## 5. Feature Engineering

Extract structured numeric features from the parsed JSON fields.

### 5a. Skills — multi-hot encoding
Build a vocabulary of the top N_SKILLS skills by frequency across all profiles.
Each profile is represented as a binary vector: 1 if the profile has that skill, 0 otherwise.
This is called multi-hot encoding (as opposed to one-hot, where only one value is 1).

### 5b. Intent text — TF-IDF
`goals`, `needs`, and `can_offer` are concatenated into a single text field per profile.
TF-IDF converts this into a numeric vector that captures what each person is looking
for and what they can contribute — the core of professional intent.

### 5c. Experience — numeric feature engineering
Rather than trying to encode the full work history, we extract two meaningful signals:
- `num_roles`: how many jobs the person has held (career breadth)
- `avg_tenure`: average years per role (career depth / stability)

### 5d. Education — ordinal encoding
Highest degree is extracted and encoded as an ordinal:
None=0, Bachelor=1, Master=2, PhD=3

In [8]:
# Step 1: build vocabulary from top N_SKILLS skills across all profiles
all_skills = []
for val in df_model["skills"]:
    parsed = safe_parse(val)
    if isinstance(parsed, list):
        seen = set()
        for s in parsed:
            if isinstance(s, str):
                clean = s.strip().lower()
                if clean and clean not in seen:
                    all_skills.append(clean)
                    seen.add(clean)

skill_vocab = [skill for skill, _ in Counter(all_skills).most_common(N_SKILLS)]
print(f"Vocabulary built: top {len(skill_vocab)} skills")
print(f"Sample: {skill_vocab[:10]}")

# Step 2: encode each profile as a binary vector over the vocabulary
skill_to_idx = {skill: i for i, skill in enumerate(skill_vocab)}

skill_matrix = np.zeros((len(df_model), len(skill_vocab)), dtype=np.float32)

for row_idx, val in enumerate(df_model["skills"]):
    parsed = safe_parse(val)
    if isinstance(parsed, list):
        seen = set()
        for s in parsed:
            if isinstance(s, str):
                clean = s.strip().lower()
                if clean in skill_to_idx and clean not in seen:
                    skill_matrix[row_idx, skill_to_idx[clean]] = 1.0
                    seen.add(clean)

print(f"\nSkill matrix shape : {skill_matrix.shape}")
print(f"Sparsity           : "
      f"{100*(1 - skill_matrix.sum()/(skill_matrix.shape[0]*skill_matrix.shape[1])):.1f}%")
print(f"Avg skills encoded : {skill_matrix.sum(axis=1).mean():.1f} per profile")

Vocabulary built: top 100 skills
Sample: ['python', 'sql', 'statistics', 'machine learning', 'tableau', 'r', 'aws', 'matlab', 'data analysis', 'react']



Skill matrix shape : (50000, 100)
Sparsity           : 93.1%
Avg skills encoded : 6.9 per profile


In [9]:
# Concatenate goals + needs + can_offer into one text string per profile
def build_intent_text(row):
    parts = []
    for col in ["goals", "needs", "can_offer"]:
        parsed = safe_parse(row[col])
        if isinstance(parsed, list):
            parts.extend([str(s).lower().strip() for s in parsed
                         if isinstance(s, str)])
    return " ".join(parts) if parts else "unknown"

intent_texts = df_model.apply(build_intent_text, axis=1).tolist()

# Fit TF-IDF — max_features caps the vocabulary at N_TFIDF_TERMS
tfidf = TfidfVectorizer(
    max_features=N_TFIDF_TERMS,
    stop_words="english",
    ngram_range=(1, 1)   # unigrams only — keeps it interpretable
)
intent_matrix = tfidf.fit_transform(intent_texts)

print(f"TF-IDF matrix shape    : {intent_matrix.shape}")
print(f"Vocabulary sample      : {tfidf.get_feature_names_out()[:15].tolist()}")

TF-IDF matrix shape    : (50000, 33)
Vocabulary sample      : ['advice', 'career', 'coaching', 'code', 'collaboration', 'connections', 'development', 'experience', 'expertise', 'feedback', 'growth', 'guidance', 'industry', 'insights', 'job']


In [10]:
def parse_experience_features(val):
    """
    Extract two numeric signals from the experience JSON field:
    - num_roles  : total number of roles held (career breadth)
    - avg_tenure : average years per role (career depth)
    
    Duration is stored as a string e.g. "2 years" or "6 months".
    We extract the numeric value and convert months to fractional years.
    """
    parsed = safe_parse(val)
    
    if not isinstance(parsed, list) or len(parsed) == 0:
        return 0, 0.0
    
    num_roles = len(parsed)
    tenures = []
    
    for job in parsed:
        if not isinstance(job, dict):
            continue
        duration_str = job.get("duration", "")
        if not duration_str:
            continue
        
        # Extract number + unit from strings like "2 years", "6 months", "1 year"
        match = re.search(r"(\d+(?:\.\d+)?)\s*(year|month)", 
                          str(duration_str).lower())
        if match:
            value = float(match.group(1))
            unit  = match.group(2)
            years = value if "year" in unit else value / 12
            tenures.append(years)
    
    avg_tenure = float(np.mean(tenures)) if tenures else 0.0
    return num_roles, avg_tenure


exp_features = df_model["experience"].apply(
    lambda v: pd.Series(parse_experience_features(v),
                        index=["num_roles", "avg_tenure"])
)

print(f"Experience features shape : {exp_features.shape}")
print(f"\nnum_roles summary:")
print(exp_features["num_roles"].describe().round(2))
print(f"\navg_tenure summary:")
print(exp_features["avg_tenure"].describe().round(2))

Experience features shape : (50000, 2)

num_roles summary:
count    50000.00
mean         3.18
std          1.80
min          1.00
25%          2.00
50%          3.00
75%          4.00
max          8.00
Name: num_roles, dtype: float64

avg_tenure summary:
count    50000.00
mean         1.77
std          0.60
min          0.50
25%          1.40
50%          1.75
75%          2.00
max          4.00
Name: avg_tenure, dtype: float64


In [11]:
DEGREE_MAP = {
    "hs":         0, "high school": 0, "none":    0,
    "bs":         1, "ba":          1, "bsc":     1, "bachelor": 1,
    "ms":         2, "ma":          2, "msc":     2, "master":   2, "mba": 2,
    "phd":        3, "doctorate":   3, "md":      3,
}

def extract_highest_degree(val):
    """
    Parse education JSON and return the highest degree level as an ordinal.
    Degree strings are matched case-insensitively against DEGREE_MAP.
    Returns 0 if no education data is present.
    """
    parsed = safe_parse(val)
    if not isinstance(parsed, list) or len(parsed) == 0:
        return 0
    
    max_degree = 0
    for edu in parsed:
        if not isinstance(edu, dict):
            continue
        degree_str = str(edu.get("degree", "")).lower().strip()
        for key, val_ord in DEGREE_MAP.items():
            if key in degree_str:
                max_degree = max(max_degree, val_ord)
                break
    return max_degree

df_model["highest_degree"] = df_model["education"].apply(extract_highest_degree)

print("Highest degree distribution:")
print(df_model["highest_degree"].value_counts().sort_index().to_string())

Highest degree distribution:
highest_degree
1    19875
2    21424
3     8701


## 6. Encode Categorical Columns

- `seniority_level` → ordinal (entry=0, mid=1, senior=2, executive=3)
- `remote_preference` → ordinal (onsite=0, hybrid=1, remote=2)
- `industry` → one-hot (pandas get_dummies)
- `current_role` → one-hot (24 unique values confirmed in EDA — direct encoding)
- `location` → extract country token → one-hot

In [12]:
# Ordinal encoding — order carries meaning so integers are correct here
# Using .map() with fill_value=0 handles any unexpected values gracefully
df_model["seniority_encoded"] = (
    df_model["seniority_level"]
    .str.lower()
    .str.strip()
    .map(SENIORITY_MAP)
    .fillna(0)
    .astype(int)
)

df_model["remote_encoded"] = (
    df_model["remote_preference"]
    .str.lower()
    .str.strip()
    .map(REMOTE_MAP)
    .fillna(1)   # default to hybrid if unknown
    .astype(int)
)

print("Seniority encoded value counts:")
print(df_model["seniority_encoded"].value_counts().sort_index().to_string())
print("\nRemote encoded value counts:")
print(df_model["remote_encoded"].value_counts().sort_index().to_string())

Seniority encoded value counts:
seniority_encoded
0    14950
1    17559
2    12393
3     5098

Remote encoded value counts:
remote_encoded
0    16559
1    16651
2    16790


In [13]:
# Group industries that appear in fewer than 0.5% of profiles into "Other"
# This prevents a large number of near-zero columns in the feature matrix
industry_threshold = int(0.005 * len(df_model))
industry_counts    = df_model["industry"].value_counts()
rare_industries    = industry_counts[industry_counts < industry_threshold].index

df_model["industry_clean"] = df_model["industry"].apply(
    lambda x: "Other" if x in rare_industries else x
)

industry_ohe = pd.get_dummies(
    df_model["industry_clean"], prefix="industry"
).astype(np.float32)

print(f"Industries kept  : {df_model['industry_clean'].nunique()}")
print(f"Industries grouped as Other: {len(rare_industries)}")
print(f"One-hot matrix shape: {industry_ohe.shape}")

Industries kept  : 15
Industries grouped as Other: 0
One-hot matrix shape: (50000, 15)


In [14]:
# 24 unique values confirmed in EDA — direct one-hot encoding
role_ohe = pd.get_dummies(
    df_model["current_role"], prefix="role"
).astype(np.float32)

print(f"Role one-hot shape : {role_ohe.shape}")
print(f"Columns            : {role_ohe.columns.tolist()}")

Role one-hot shape : (50000, 24)
Columns            : ['role_Backend Developer', 'role_Business Analyst', 'role_CTO', 'role_Cloud Engineer', 'role_Data Analyst', 'role_Data Engineer', 'role_Data Scientist', 'role_DevOps Engineer', 'role_Engineering Manager', 'role_Frontend Developer', 'role_Full Stack Developer', 'role_ML Engineer', 'role_Platform Engineer', 'role_Product Manager', 'role_QA Engineer', 'role_Research Scientist', 'role_Scrum Master', 'role_Security Engineer', 'role_Site Reliability Engineer', 'role_Software Engineer', 'role_Solutions Architect', 'role_Systems Architect', 'role_Technical Lead', 'role_UI/UX Designer']


In [15]:
# 18 unique values confirmed via EDA — direct one-hot encoding
company_ohe = pd.get_dummies(
    df_model["current_company"], prefix="company"
).astype(np.float32)

print(f"Company one-hot shape : {company_ohe.shape}")
print(f"Companies encoded     : {df_model['current_company'].unique().tolist()}")

Company one-hot shape : (50000, 20)
Companies encoded     : ['Meta', 'Deloitte', 'McKinsey', 'Microsoft', 'Wells Fargo', 'Amazon', 'JP Morgan', 'Goldman Sachs', 'Cisco', 'Oracle', 'Tesla', 'Netflix', 'Accenture', 'Salesforce', 'Apple', 'Adobe', 'Intel', 'Google', 'SAP', 'IBM']


In [16]:
def extract_region(location_str):
    """
    Extract a coarse region token from a location string.
    e.g. "East William, AK" → "US"
       "London, UK"         → "UK"
    
    US states are detected by their 2-letter abbreviation after a comma.
    Everything else falls through to "Other".
    """
    if not isinstance(location_str, str) or location_str == "unknown":
        return "Other"
    
    us_states = {
        "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID",
        "IL","IN","IA","KS","KY","LA","ME","MD","MA","MI","MN","MS",
        "MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK",
        "OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV",
        "WI","WY","DC"
    }
    
    parts = [p.strip() for p in location_str.split(",")]
    
    # Check for US state abbreviation
    if len(parts) >= 2 and parts[-1].upper() in us_states:
        return "US"
    
    # Check last token for known country codes / names
    last = parts[-1].upper()
    known = {"UK": "UK", "CANADA": "Canada", "INDIA": "India",
             "AUSTRALIA": "Australia", "GERMANY": "Germany",
             "FRANCE": "France", "SINGAPORE": "Singapore"}
    for key, val in known.items():
        if key in last:
            return val
    
    return "Other"

df_model["region"] = df_model["location"].apply(extract_region)

print("Region distribution:")
print(df_model["region"].value_counts().to_string())

location_ohe = pd.get_dummies(
    df_model["region"], prefix="region"
).astype(np.float32)

print(f"\nLocation one-hot shape: {location_ohe.shape}")

Region distribution:
region
US           30000
UK            2567
Australia     2528
Germany       2521
Canada        2500
India         2494
France        2470
Other         2465
Singapore     2455

Location one-hot shape: (50000, 9)


## 7. Scale Numeric Features

StandardScaler normalises each numeric column to mean=0, std=1.
This ensures no single numeric feature dominates the distance
calculations in K-Means simply because of its scale.

`connections` receives a log1p transform first because it is
right-skewed — this was confirmed during EDA in notebook 01.
Without it, a few profiles with 10,000+ connections would
distort the scaled values for everyone else.

In [17]:
scaler = StandardScaler()

numeric_features = pd.DataFrame({
    "years_experience": df_model["years_experience"].values,
    "connections":      np.log1p(df_model["connections"].values),
    "seniority":        df_model["seniority_encoded"].values,
    "remote":           df_model["remote_encoded"].values,
    "highest_degree":   df_model["highest_degree"].values,
    "num_roles":        exp_features["num_roles"].values,
    "avg_tenure":       exp_features["avg_tenure"].values,
})

numeric_scaled = scaler.fit_transform(numeric_features).astype(np.float32)

print(f"Numeric block shape : {numeric_scaled.shape}")
print(f"\nFeature means after scaling (should all be ~0):")
print(pd.Series(numeric_scaled.mean(axis=0),
                index=numeric_features.columns).round(4).to_string())

Numeric block shape : (50000, 7)

Feature means after scaling (should all be ~0):
years_experience    0.0
connections         0.0
seniority          -0.0
remote             -0.0
highest_degree     -0.0
num_roles           0.0
avg_tenure         -0.0


## 8. Assemble Feature Matrix

All encoded blocks are concatenated horizontally using `scipy.sparse.hstack`.
The sparse format is used because the skill matrix and TF-IDF matrix are
both sparse — storing them as dense arrays would waste significant memory.

Final matrix structure:
| Block | Columns | Type |
|---|---|---|
| Skills multi-hot | 100 | dense → sparse |
| Intent TF-IDF | 50 | sparse |
| Industry one-hot | N | dense → sparse |
| Role one-hot | 24 | dense → sparse |
| Location one-hot | N | dense → sparse |
| Numeric (scaled) | 7 | dense → sparse |

In [18]:
X = hstack([
    csr_matrix(skill_matrix * 2.0),
    intent_matrix * 1.5,
    csr_matrix(industry_ohe.values * 0.5),
    csr_matrix(role_ohe.values * 1.5),
    csr_matrix(location_ohe.values * 0.5),
    csr_matrix(numeric_scaled * 1.5),
]).astype(np.float32)

print(f"Final feature matrix shape : {X.shape}")
print(f"Total features             : {X.shape[1]}")
print(f"Memory (sparse)            : "
      f"{X.data.nbytes / 1024 / 1024:.1f} MB")

# Break down where the features come from
print(f"\nFeature block breakdown:")
print(f"  Skills multi-hot  : {skill_matrix.shape[1]} (weight=2.0)")
print(f"  Intent TF-IDF     : {intent_matrix.shape[1]} (weight=1.5)")
print(f"  Industry one-hot  : {industry_ohe.shape[1]} (weight=0.5)")
print(f"  Role one-hot      : {role_ohe.shape[1]} (weight=1.5)")
print(f"  Location one-hot  : {location_ohe.shape[1]} (weight=0.5)")
print(f"  Numeric scaled    : {numeric_scaled.shape[1]} (weight=1.5)")

Final feature matrix shape : (50000, 188)
Total features             : 188
Memory (sparse)            : 5.2 MB

Feature block breakdown:
  Skills multi-hot  : 100 (weight=2.0)
  Intent TF-IDF     : 33 (weight=1.5)
  Industry one-hot  : 15 (weight=0.5)
  Role one-hot      : 24 (weight=1.5)
  Location one-hot  : 9 (weight=0.5)
  Numeric scaled    : 7 (weight=1.5)


## 9. Save Outputs

In [19]:
import os
from src.config import PROCESSED_PATH

# Save feature matrix — convert to dense for numpy save
# (for 50k profiles this is manageable in memory)
X_dense = X.toarray()
np.save(PROCESSED_PATH, X_dense)
print(f"✓ Feature matrix saved  → {PROCESSED_PATH}")
print(f"  Shape: {X_dense.shape}")
print(f"  Size : {X_dense.nbytes / 1024 / 1024:.1f} MB")

# Save intermediate cleaned dataframe for inspection
df_model_path = os.path.join(
    os.path.dirname(PROCESSED_PATH), "df_model.csv"
)
df_model.to_csv(df_model_path, index=False)
print(f"✓ Cleaned dataframe saved → {df_model_path}")

# Save skill vocabulary so notebook 03 can use feature names
vocab_path = os.path.join(
    os.path.dirname(PROCESSED_PATH), "skill_vocab.txt"
)
with open(vocab_path, "w") as f:
    for skill in skill_vocab:
        f.write(skill + "\n")
print(f"✓ Skill vocabulary saved  → {vocab_path}")

# Save TF-IDF feature names
tfidf_path = os.path.join(
    os.path.dirname(PROCESSED_PATH), "tfidf_terms.txt"
)
with open(tfidf_path, "w") as f:
    for term in tfidf.get_feature_names_out():
        f.write(term + "\n")
print(f"✓ TF-IDF terms saved      → {tfidf_path}")

✓ Feature matrix saved  → /Users/yugjain/Documents/ML_ASS/unsupervised-professional-matching/data/processed/X_processed.npy
  Shape: (50000, 188)
  Size : 35.9 MB


✓ Cleaned dataframe saved → /Users/yugjain/Documents/ML_ASS/unsupervised-professional-matching/data/processed/df_model.csv
✓ Skill vocabulary saved  → /Users/yugjain/Documents/ML_ASS/unsupervised-professional-matching/data/processed/skill_vocab.txt
✓ TF-IDF terms saved      → /Users/yugjain/Documents/ML_ASS/unsupervised-professional-matching/data/processed/tfidf_terms.txt


## 10. Preprocessing Summary

| Step | Input | Output | Notes |
|---|---|---|---|
| Drop columns | 20 cols | 16 cols | PII + 4 low-signal cols removed |
| Null filling | raw nulls | 0 nulls | median for numeric, "unknown" for cat |
| Skills multi-hot | JSON strings | (50000, 100) | top 100 skills, deduplicated |
| Intent TF-IDF | 3 text fields | (50000, 50) | goals + needs + can_offer |
| Experience | JSON strings | (50000, 2) | num_roles + avg_tenure |
| Education | JSON strings | (50000,) | highest degree ordinal |
| Seniority | 4 levels | (50000,) | ordinal 0–3 |
| Remote pref | 3 levels | (50000,) | ordinal 0–2 |
| Industry | N categories | (50000, N) | one-hot, rare grouped as Other |
| Role | 24 categories | (50000, 24) | one-hot direct |
| Location | free text | (50000, N) | region extracted, one-hot |
| Numeric scaling | 7 features | (50000, 7) | StandardScaler |
| **Final matrix** | all blocks | **(50000, ~200)** | hstack, saved as .npy |

**Next:** `03_clustering.ipynb` — PCA, K-Means, elbow method, silhouette score,
cluster profiling, and similarity-based recommendation demo.